# Competition networks

In [1]:
import pandas as pd
import json
import seaborn as sns
import numpy as np
import networkx as nx
from pyvis.network import Network

import sys
sys.path.append("../functions/")
import networks

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
media_db = pd.read_csv("https://raw.githubusercontent.com/cdanielmachado/carveme/refs/heads/master/carveme/data/input/media_db.tsv",sep="\t")
compounds = media_db[media_db.medium=="M9"].compound.values
compounds = list(compounds)
compounds.remove("glc__D")
m9_compounds = ["M_"+met+"__91__e__93__" for met in compounds]

In [4]:
with open("../data/Venturelli2018/generated/metid2metname.json") as json_file:
    metid2metname = json.load(json_file)

results = pd.read_csv("../output/Venturelli2018/results_steadiercom/NONE_abundance.tsv",sep="\t")

results["mass_rate*frequency"] = results["frequency"]*results["mass_rate"]
results = results[~results.compound.isin(m9_compounds)]

results = results[results.frequency>0.1]
results = results[results["mass_rate*frequency"]>1e-5]

In [5]:
results[(results.donor!="environment") & (results.receiver!="environment")].sort_values("rate").tail(80)

,donor,receiver,compound,mass_rate,rate,frequency,community,medium,mass_rate*frequency
1176,Bacteroides_ovatus_ATCC_8483,Bacteroides_thetaiotaomicron_VPI_5482,M_mqn8__91__e__93__,0.000026,0.000036,0.997,NONE,LB_extend,0.000026
1160,Bacteroides_ovatus_ATCC_8483,Bacteroides_thetaiotaomicron_VPI_5482,M_thymd__91__e__93__,0.000028,0.000114,0.967,NONE,LB_extend,0.000027
1174,Bacteroides_uniformis_ATCC_8492,Bacteroides_vulgatus_ATCC_8482,M_acgam__91__e__93__,0.000026,0.000117,0.989,NONE,LB_extend,0.000026
1151,Faecalibacterium_prausnitzii_A2_165,Bacteroides_thetaiotaomicron_VPI_5482,M_thymd__91__e__93__,0.000029,0.000119,0.967,NONE,LB_extend,0.000028
1100,Bacteroides_ovatus_ATCC_8483,Clostridium_hiranonis_TO_931_DSM_13275,M_adn__91__e__93__,0.000036,0.000135,0.708,NONE,LB_extend,0.000026
...,...,...,...,...,...,...,...,...,...
83,Clostridium_hiranonis_TO_931_DSM_13275,Bacteroides_thetaiotaomicron_VPI_5482,M_7dhcdchol__91__e__93__,0.006664,0.017107,0.196,NONE,LB_extend,0.001306
66,Faecalibacterium_prausnitzii_A2_165,Bacteroides_ovatus_ATCC_8483,M_fru__91__e__93__,0.008556,0.047494,0.450,NONE,LB_extend,0.003850
62,Faecalibacterium_prausnitzii_A2_165,Bacteroides_thetaiotaomicron_VPI_5482,M_fru__91__e__93__,0.008648,0.048003,0.492,NONE,LB_extend,0.004255
49,Bacteroides_uniformis_ATCC_8492,Bacteroides_ovatus_ATCC_8483,M_fru__91__e__93__,0.010126,0.056208,0.497,NONE,LB_extend,0.005033


## With dataset 4 data

In [6]:
import matplotlib as mpl
# Color end points for colorFader
b1='#f8cdcf' #bright pink
b2='#fa4f56' #red

c1='#ffffff' #white
c2='#0b5394' #blue
    
## From https://stackoverflow.com/questions/25668828/how-to-create-colour-gradient-in-python
def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)





In [7]:
def competition_network(network_data,network_exp,closeness_centrality_met,cutoff=0):
    net = Network()
    
    maximum_centrality = pd.Series(closeness_centrality_met).max()

    nodes = list(set(list(network_data.index.get_level_values(0).unique())+list(network_data.index.get_level_values(1).unique())))
    
    
    if isinstance(closeness_centrality_met,pd.Series):
        value = [(closeness_centrality_met[node]*10) for node in nodes]
        color = [colorFader(b1,b2,closeness_centrality_met[node]/maximum_centrality) for node in nodes]
        net.add_nodes(nodes,color=color,value=value)
    else:
        net.add_nodes(nodes)
    maximum_overlap = network_data.max()
    
    i=0
    j=0
    
    for index,value in network_data.items():
        new_index1 = index[0].split("_")[0][0].upper()+index[0].split("_")[1][0].upper()
        new_index2 = index[1].split("_")[0][0].upper()+index[1].split("_")[1][0].upper()
        
        new_index = (new_index1,new_index2)
        new_index_rev = (new_index2,new_index1)



        if value>cutoff:
            if (new_index in network_exp.index) or (new_index_rev in network_exp.index):
                color = "red"
                i+=1
            else:
                color = colorFader(c1,c2,value/maximum_overlap)
            net.add_edge(index[0],index[1],width=value*2,color=color)
            j+=1
        
    net.barnes_hut(gravity=-1000, central_gravity=0.1, spring_length=150, spring_strength=0.0015, damping=0.09, overlap=0.8)

    print(f"Matching exp for {(i/j)*100}% of interactions")
    
    template = net.templateEnv.get_template(net.path)
    net.template = template
    return net


In [8]:
dataset4 = pd.read_excel("../data/Venturelli2018/msb178157-sup-0005-datasetev4.xlsx",sheet_name="Dataset EV4",index_col=0)
dataset4.columns = dataset4.columns.map(lambda x: x.replace("'",""))
dataset4.columns = dataset4.columns.map(lambda x: x.replace(" ",""))

### Overlap count (compound consumed with high frequency)

In [9]:
results_env = results[(results.donor=="environment")]

**Count (cutoff frequency=0.5!!)**

In [10]:
network_data = networks.competition_network_data(results_env,freq_cutoff=0.5)

**Network**

In [11]:
consumption = dataset4<=-1
consumption = consumption.map(int)

network_cons = consumption.T.dot(consumption)
network_cons = network_cons.mask(np.tril(np.ones(network_cons.shape)).astype(np.bool)).stack() 
network_cons = network_cons[network_cons>1]

#centrality_metrics_SP = networks.competition_centrality_metrics(network_data,cutoff=4).sort_values("closeness (1/data)")

In [13]:
#closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = competition_network(network_data,network_cons,closeness_centrality_met=False,cutoff=4)
net.save_graph("figures/network_member_member_competition.html")

net.show("network_file.html")

Matching exp for 88.88888888888889% of interactions
network_file.html


### Network

In [14]:
consumption = dataset4<=-2
consumption = consumption.map(int)

network_cons = consumption.T.dot(consumption)
network_cons = network_cons.mask(np.tril(np.ones(network_cons.shape)).astype(np.bool)).stack() 
network_cons = network_cons[network_cons>1]

In [15]:
#centrality_metrics_SP = networks.competition_centrality_metrics(network_data,cutoff=7).sort_values("closeness (1/data)")

In [17]:
#closeness_centrality_met = centrality_metrics_SP["closeness (1/data)"].to_dict()

net = competition_network(network_data,network_cons,closeness_centrality_met=False,cutoff=4)
net.show("network_file.html")

Matching exp for 70.37037037037037% of interactions
network_file.html
